In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 88.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


## Local Inference on GPU 
Model page: https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 9.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.1 MB/s eta 0:00:00:00:0100:01


In [22]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os

In [6]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [7]:
import torch
from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(model_name)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loaded successfully!


In [8]:
import base64
from io import BytesIO
from PIL import Image

image = Image.open("/kaggle/input/datasets/ramakanthjogada/pope-dataset/images/images/000000006578.jpg").convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Is there a person in this image? Answer only Yes or No."}
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

response = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

print(response[0])

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Yes.


In [25]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "/images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_name"] == target_name:
            return record["query"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [12]:
def evaluate(query_text, b64_str):
    # Predict
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": query_text + "\nAnswer only Yes or No."}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    response = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    return response[0]

In [14]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [15]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result



In [16]:
def evaluate_standard_blur(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Apply Standard Blur
    blur_standard = cv2.GaussianBlur(img_rgb, (15, 15), 0)

    # Encode
    b64_str = get_base64_from_rgb(blur_standard)
    query_text = query_from_path(image_path)

    # Predict
    result = evaluate(query_text,b64_str)
    return result

In [17]:
def evaluate_heavy_blur(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Apply Heavy Blur
    blur_heavy = cv2.GaussianBlur(img_rgb, (51, 51), 0)

    # Encode
    b64_str = get_base64_from_rgb(blur_heavy)
    query_text = query_from_path(image_path)

    # Predict
    result = evaluate(query_text,b64_str)
    return result

In [18]:
def evaluate_occlusion(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Apply Occlusion (Central 50% gray box)
    occlusion = img_rgb.copy()
    h, w, _ = occlusion.shape
    c_h, c_w = h // 2, w // 2
    dy, dx = h // 4, w // 4
    occlusion[c_h - dy : c_h + dy, c_w - dx : c_w + dx] = [128, 128, 128]

    # Encode
    b64_str = get_base64_from_rgb(occlusion)
    query_text = query_from_path(image_path)

    # Predict
    result = evaluate(query_text,b64_str)
    return result

In [19]:
def evaluate_jpeg_compression(image_path):
    img = cv2.imread(image_path)

    # Apply JPEG Compression (Quality = 5)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 5]
    _, encimg = cv2.imencode('.jpg', img, encode_param)
    img_compressed = cv2.imdecode(encimg, 1)
    img_compressed_rgb = cv2.cvtColor(img_compressed, cv2.COLOR_BGR2RGB)

    # Encode
    b64_str = get_base64_from_rgb(img_compressed_rgb)
    query_text = query_from_path(image_path)

    # Predict
    result = evaluate(query_text,b64_str)
    return result

In [20]:
def evaluate_resizing(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Apply Resizing (10% resolution)
    h, w, _ = img_rgb.shape
    small_img = cv2.resize(img_rgb, (w // 10, h // 10), interpolation=cv2.INTER_LINEAR)
    img_resized_rgb = cv2.resize(small_img, (w, h), interpolation=cv2.INTER_NEAREST)

    # Encode
    b64_str = get_base64_from_rgb(img_resized_rgb)
    query_text = query_from_path(image_path)

    # Predict
    result = evaluate(query_text,b64_str)
    return result

In [26]:
import os
import pandas as pd
from tqdm.notebook import tqdm

# --- Paths Configuration (Adjust as needed) ---
IMAGE_DIR = "/kaggle/input/datasets/ramakanthjogada/pope-dataset/images/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/pope-dataset/annotations.json"

# Load annotations lookup map for faster access
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Create a lookup map for ground truth: (image_id) -> label
# Since we are evaluating the actual files in /images/, we target '/images/' records
ground_truth_map = {}
for record in annotations_list:
    if record["image_name"].startswith("/images/"):
        img_id = record["image_id"]
        ground_truth_map[img_id] = record["label"]

# Get list of all images in the directory
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

results = []

print(f"Processing {len(image_files)} images through all variations...")
for filename in tqdm(image_files):
    # Extract image_id from filename (e.g., '000000046207.jpg' -> 46207)
    try:
        image_id = int(os.path.splitext(filename)[0])
    except ValueError:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    ground_truth = ground_truth_map.get(image_id, "Unknown")

    # Evaluate under all conditions
    try:
        orig_val = evaluate_original(image_path)
    except Exception as e:
        orig_val = f"Error: {str(e)}"

    try:
        std_blur_val = evaluate_standard_blur(image_path)
    except Exception as e:
        std_blur_val = f"Error: {str(e)}"

    try:
        heavy_blur_val = evaluate_heavy_blur(image_path)
    except Exception as e:
        heavy_blur_val = f"Error: {str(e)}"

    try:
        occlusion_val = evaluate_occlusion(image_path)
    except Exception as e:
        occlusion_val = f"Error: {str(e)}"

    try:
        jpeg_val = evaluate_jpeg_compression(image_path)
    except Exception as e:
        jpeg_val = f"Error: {str(e)}"

    try:
        resize_val = evaluate_resizing(image_path)
    except Exception as e:
        resize_val = f"Error: {str(e)}"

    # Store record
    results.append({
        "Image_Id": image_id,
        "Ground_truth": ground_truth,
        "Original": orig_val,
        "Standard_Blur": std_blur_val,
        "Heavy_Blur": heavy_blur_val,
        "Occlusion": occlusion_val,
        "JPEG_Compression": jpeg_val,
        "Resizing": resize_val
    })

# Create final DataFrame
df_results = pd.DataFrame(results)
display(df_results.head(10))

# Save to working directory
df_results.to_csv("evaluation_results.csv", index=False)
print("Saved results to evaluation_results.csv")

Processing 500 images through all variations...


  0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


,Image_Id,Ground_truth,Original,Standard_Blur,Heavy_Blur,Occlusion,JPEG_Compression,Resizing
0,169361,No,No,No,No,No,No,No
1,47022,No,No,No,No,No,No,No
2,556064,No,No,No,No,No,No,No
3,474004,No,No,No,Yes.,No,No,No
4,306710,No,No,No,No,No,No,Yes.
5,33882,No,No,No,No,No,No,Yes.
6,195213,No,No,Yes,Yes,No,Yes,Yes.
7,196545,No,No,No,Yes.,No,No,Yes.
8,186703,No,No,No,No,No,No,Yes.
9,215863,No,No,No,No,No,No,No


Saved results to evaluation_results.csv
